# 24 — Directed K_nm + FIM Strange Loop at N=16

Notebook 17 showed flat symmetric K_nm fails at N=16 (R<0.7 at K_scale=12).
Notebook 20 showed FIM creates hysteresis (width 0.61) but stays 2nd order.
Notebook 19 showed cross-scale coupling IS directed (TE asymmetry 0.36).

**Question:** Does directed (asymmetric) coupling + FIM strange loop solve N=16?

2x2 comparison: {symmetric, directed} x {no FIM, FIM}

In [ ]:
import numpy as np
import json
from pathlib import Path

import sys
sys.path.insert(0, '/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/src')
from scpn_quantum_control.bridge.knm_hamiltonian import build_knm_paper27, OMEGA_N_16

N = 16
K_sym = build_knm_paper27(L=N)
omega = OMEGA_N_16[:N]
print(f'N={N}, K_sym shape={K_sym.shape}, omega range=[{omega.min():.3f}, {omega.max():.3f}]')

In [ ]:
def build_directed_knm(K_sym, asymmetry=0.361):
    """Build directed K_nm from symmetric + TE-like asymmetry.
    
    Lower indices (autonomic scales) drive upper (cortical scales).
    K_dir[i,j] = K_sym[i,j] * (1 + asymmetry) for i < j (upward)
    K_dir[j,i] = K_sym[j,i] * (1 - asymmetry) for j > i (downward)
    """
    K_dir = K_sym.copy()
    n = K_sym.shape[0]
    for i in range(n):
        for j in range(i + 1, n):
            K_dir[i, j] = K_sym[i, j] * (1 + asymmetry)
            K_dir[j, i] = K_sym[j, i] * max(1 - asymmetry, 0.01)
    return K_dir

def fim_gradient(phases, i):
    """FIM strange loop gradient for oscillator i.
    
    The system observes its own collective state (order parameter)
    and each oscillator adjusts to align with the collective phase.
    The Fisher Information term provides sensitivity-dependent pull.
    """
    N = len(phases)
    z = np.exp(1j * phases)
    mu = np.angle(np.mean(z))
    R = np.abs(np.mean(z))
    phase_diff = (mu - phases[i] + np.pi) % (2 * np.pi) - np.pi
    # Fisher sensitivity: diverges as R→1 (near-sync is highly sensitive)
    sensitivity = 1.0 / (1.0 - R**2 + 0.01)
    return (1.0 / N) * np.sin(phase_diff) * min(sensitivity, 50.0)

def simulate_kuramoto(N, K, omega, K_scale=1.0, fim_lambda=0.0,
                       dt=0.02, T=100.0, noise=0.05, seed=42):
    """Simulate Kuramoto with optional directed coupling and FIM lift.
    
    Returns final R (averaged over last 25% of trajectory).
    """
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2 * np.pi, N)
    n_steps = int(T / dt)
    K_eff = K * K_scale
    
    R_tail = []
    for s in range(n_steps):
        # Coupling term: sum_j K[i,j] sin(theta_j - theta_i)
        diff = theta[None, :] - theta[:, None]  # diff[i,j] = theta[j] - theta[i]
        coupling = np.sum(K_eff * np.sin(diff), axis=1) / N
        
        dphi = omega + coupling
        
        # FIM strange loop term
        if fim_lambda > 0:
            for i in range(N):
                dphi[i] += fim_lambda * fim_gradient(theta, i)
        
        theta += dt * dphi + np.sqrt(dt) * noise * rng.normal(size=N)
        theta %= 2 * np.pi
        
        # Collect tail for averaging
        if s >= n_steps * 3 // 4:
            R = float(np.abs(np.mean(np.exp(1j * theta))))
            R_tail.append(R)
    
    return float(np.mean(R_tail))

# Build directed K_nm
K_dir = build_directed_knm(K_sym, asymmetry=0.361)

# Quick test
R_test = simulate_kuramoto(N, K_sym, omega, K_scale=1.0, fim_lambda=0.0)
print(f'Test (symmetric, no FIM, K_scale=1): R={R_test:.4f}')
print('Functions defined.')

## 2x2 Comparison: {Symmetric, Directed} x {No FIM, FIM}

Sweep K_scale from 1 to 20 for all 4 configurations.

In [ ]:
K_scales = np.array([1, 2, 3, 4, 5, 6, 8, 10, 12, 14, 16, 18, 20], dtype=float)
fim_lambdas = [0.0, 1.0]
K_matrices = {'symmetric': K_sym, 'directed': K_dir}

# Run all 4 configurations
results_2x2 = {}
for k_name, K_mat in K_matrices.items():
    for fim_lam in fim_lambdas:
        config_name = f'{k_name}_fim{fim_lam:.1f}'
        print(f'\n=== {config_name} ===')
        R_values = []
        for K_scale in K_scales:
            # Average over 3 seeds for robustness
            R_seeds = []
            for seed in [42, 123, 456]:
                R = simulate_kuramoto(N, K_mat, omega, K_scale=K_scale,
                                       fim_lambda=fim_lam, dt=0.02, T=100.0,
                                       noise=0.05, seed=seed)
                R_seeds.append(R)
            R_mean = np.mean(R_seeds)
            R_values.append(R_mean)
            print(f'  K_scale={K_scale:5.1f}: R={R_mean:.4f} (seeds: {[round(r,3) for r in R_seeds]})')
        results_2x2[config_name] = R_values

# Print comparison table
print('\n\n=== COMPARISON TABLE ===')
print(f'{"K_scale":>8}', end='')
for config in results_2x2:
    print(f' {config:>20}', end='')
print()
for i, K_scale in enumerate(K_scales):
    print(f'{K_scale:8.1f}', end='')
    for config in results_2x2:
        print(f' {results_2x2[config][i]:20.4f}', end='')
    print()

In [ ]:
# Analysis: which configuration achieves R > 0.8 at lowest K_scale?
print('=== SYNC THRESHOLD ANALYSIS ===')
R_threshold = 0.8
for config_name, R_values in results_2x2.items():
    R_arr = np.array(R_values)
    idx = np.where(R_arr >= R_threshold)[0]
    if len(idx) > 0:
        K_sync = float(K_scales[idx[0]])
        print(f'{config_name}: R>={R_threshold} at K_scale={K_sync}')
    else:
        max_R = float(R_arr.max())
        print(f'{config_name}: NEVER reaches R>={R_threshold} (max R={max_R:.4f} at K={K_scales[R_arr.argmax()]:.1f})')

# Determine winning configuration
print('\n=== VERDICT ===')
best_config = None
best_K = float('inf')
for config_name, R_values in results_2x2.items():
    R_arr = np.array(R_values)
    idx = np.where(R_arr >= R_threshold)[0]
    if len(idx) > 0 and K_scales[idx[0]] < best_K:
        best_K = K_scales[idx[0]]
        best_config = config_name

if best_config:
    print(f'Best: {best_config} (R>={R_threshold} at K_scale={best_K})')
    if 'directed' in best_config and 'fim1' in best_config:
        print('DIRECTED + FIM wins! Both ingredients needed.')
    elif 'directed' in best_config:
        print('Direction alone is sufficient.')
    elif 'fim1' in best_config:
        print('FIM alone is sufficient.')
    else:
        print('Symmetric coupling without FIM is sufficient.')
else:
    print(f'No configuration reaches R>={R_threshold} at N={N}.')
    print('N=16 sync remains an open problem.')
    # Check max R achieved
    for config_name, R_values in results_2x2.items():
        print(f'  {config_name}: max R={max(R_values):.4f}')

In [ ]:
# Additional: sweep FIM lambda for best directed K_scale
print('\n=== FIM LAMBDA SWEEP (directed, K_scale=12) ===')
lambdas = [0.0, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
for lam in lambdas:
    R_seeds = []
    for seed in [42, 123, 456]:
        R = simulate_kuramoto(N, K_dir, omega, K_scale=12.0,
                               fim_lambda=lam, dt=0.02, T=100.0,
                               noise=0.05, seed=seed)
        R_seeds.append(R)
    print(f'  lambda={lam:5.1f}: R={np.mean(R_seeds):.4f} +/- {np.std(R_seeds):.4f}')

In [ ]:
# Save results
output = {
    'experiment': 'Directed K_nm + FIM strange loop at N=16',
    'N': N,
    'asymmetry': 0.361,
    'dt': 0.02,
    'T': 100.0,
    'noise': 0.05,
    'K_scales': [float(k) for k in K_scales],
    'configs': {name: [round(float(r), 4) for r in vals]
                for name, vals in results_2x2.items()},
}

out_path = Path('/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/directed_knm_fim_n16_2026-03-29.json')
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'Results saved to {out_path}')